# Multi-Armed Bandits

This notebook explores the **multi-armed bandit problem**, one of the simplest reinforcement learning settings.

We have $k$ arms (actions), each with an unknown reward distribution. At each timestep we pull one arm and observe a reward. The goal is to maximize cumulative reward over time.

We implement and compare three classic strategies:
1. **Epsilon-Greedy** -- exploit the best known arm most of the time, explore randomly with probability $\epsilon$
2. **Upper Confidence Bound (UCB)** -- pick the arm with the highest optimistic estimate
3. **Thompson Sampling** -- Bayesian approach using posterior sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## 1. The 10-Armed Bandit Environment

Each arm $a$ has a true mean reward $q_*(a)$ drawn from $\mathcal{N}(0, 1)$.
When we pull arm $a$, we receive reward $R \sim \mathcal{N}(q_*(a), 1)$.

In [ ]:
class MultiArmedBandit:
    """A k-armed bandit with Gaussian rewards."""
    
    def __init__(self, k=10):
        self.k = k
        self.true_values = np.random.randn(k)  # true q*(a)
        self.optimal_action = np.argmax(self.true_values)
    
    def pull(self, action):
        """Return a noisy reward for the given action."""
        return np.random.randn() + self.true_values[action]


# Visualize the true reward distributions
bandit = MultiArmedBandit(k=10)

fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(bandit.k)
ax.bar(positions, bandit.true_values, color=sns.color_palette('viridis', bandit.k))
ax.set_xlabel('Arm')
ax.set_ylabel('True Reward q*(a)')
ax.set_title('True Reward Values for Each Arm')
ax.set_xticks(positions)
plt.tight_layout()
plt.show()

print(f'Optimal arm: {bandit.optimal_action} with value {bandit.true_values[bandit.optimal_action]:.3f}')

## 2. Strategy Implementations

In [ ]:
class EpsilonGreedy:
    """Epsilon-greedy strategy."""
    
    def __init__(self, k, epsilon=0.1):
        self.k = k
        self.epsilon = epsilon
        self.q_estimates = np.zeros(k)   # estimated value of each arm
        self.action_counts = np.zeros(k) # how many times each arm was pulled
    
    def select_action(self):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.k)  # explore
        else:
            return np.argmax(self.q_estimates)  # exploit
    
    def update(self, action, reward):
        self.action_counts[action] += 1
        # Incremental mean update
        self.q_estimates[action] += (
            (reward - self.q_estimates[action]) / self.action_counts[action]
        )


class UCB:
    """Upper Confidence Bound strategy."""
    
    def __init__(self, k, c=2.0):
        self.k = k
        self.c = c
        self.q_estimates = np.zeros(k)
        self.action_counts = np.zeros(k)
        self.t = 0
    
    def select_action(self):
        self.t += 1
        # Pull each arm once first
        for a in range(self.k):
            if self.action_counts[a] == 0:
                return a
        # UCB formula
        ucb_values = self.q_estimates + self.c * np.sqrt(
            np.log(self.t) / self.action_counts
        )
        return np.argmax(ucb_values)
    
    def update(self, action, reward):
        self.action_counts[action] += 1
        self.q_estimates[action] += (
            (reward - self.q_estimates[action]) / self.action_counts[action]
        )


class ThompsonSampling:
    """Thompson Sampling with Gaussian prior (known variance=1)."""
    
    def __init__(self, k):
        self.k = k
        # Gaussian prior: N(mu, 1/tau) for each arm
        self.mu = np.zeros(k)       # posterior mean
        self.tau = np.ones(k)       # posterior precision (starts at 1)
    
    def select_action(self):
        # Sample from the posterior of each arm
        samples = np.array([
            np.random.normal(self.mu[a], 1.0 / np.sqrt(self.tau[a]))
            for a in range(self.k)
        ])
        return np.argmax(samples)
    
    def update(self, action, reward):
        # Bayesian update for Gaussian with known variance=1
        self.tau[action] += 1.0  # precision increases by 1 (since variance=1)
        self.mu[action] += (reward - self.mu[action]) / self.tau[action]

## 3. Run the Experiment

We run 2000 steps on the same bandit, averaged over 500 independent runs for smooth curves.

In [ ]:
def run_experiment(bandit_class_args, strategy_factory, n_steps=2000, n_runs=500):
    """Run a bandit experiment and return per-step rewards and actions."""
    all_rewards = np.zeros((n_runs, n_steps))
    all_actions = np.zeros((n_runs, n_steps), dtype=int)
    
    for run in range(n_runs):
        bandit = MultiArmedBandit(**bandit_class_args)
        strategy = strategy_factory(bandit.k)
        
        for t in range(n_steps):
            action = strategy.select_action()
            reward = bandit.pull(action)
            strategy.update(action, reward)
            all_rewards[run, t] = reward
            all_actions[run, t] = (action == bandit.optimal_action)
    
    return all_rewards, all_actions


# Define strategies
strategies = {
    'Epsilon-Greedy (eps=0.1)': lambda k: EpsilonGreedy(k, epsilon=0.1),
    'UCB (c=2)': lambda k: UCB(k, c=2.0),
    'Thompson Sampling': lambda k: ThompsonSampling(k),
}

n_steps = 2000
n_runs = 500
results = {}

for name, factory in strategies.items():
    print(f'Running {name}...')
    rewards, optimal = run_experiment({'k': 10}, factory, n_steps, n_runs)
    results[name] = {'rewards': rewards, 'optimal': optimal}

print('Done!')

## 4. Cumulative Reward Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette('Set2', len(results))

for (name, data), color in zip(results.items(), colors):
    cumulative = np.cumsum(data['rewards'].mean(axis=0))
    ax.plot(cumulative, label=name, color=color, linewidth=2)

ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Average Cumulative Reward', fontsize=12)
ax.set_title('Cumulative Reward Over Time', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 5. Optimal Action Selection Rate

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for (name, data), color in zip(results.items(), colors):
    # Smoothed optimal action rate
    optimal_rate = data['optimal'].mean(axis=0)
    # Apply running average for smoother curves
    window = 50
    smoothed = np.convolve(optimal_rate, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=name, color=color, linewidth=2)

ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('% Optimal Action', fontsize=12)
ax.set_title('Optimal Action Selection Rate Over Time', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 6. Regret Curves

**Regret** at step $t$ = (reward of optimal arm) - (reward actually received).  
**Cumulative regret** = total opportunity cost of not always picking the best arm.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for (name, data), color in zip(results.items(), colors):
    mean_rewards = data['rewards'].mean(axis=0)
    # The optimal expected reward per step (across all runs, the true max varies;
    # we approximate with the mean reward of always-optimal action)
    # For a fair comparison, use the theoretical best: E[q*(a*)] for N(0,1) max of 10 ~ 1.54
    optimal_expected = 1.539  # E[max of 10 standard normals]
    per_step_regret = optimal_expected - mean_rewards
    cumulative_regret = np.cumsum(per_step_regret)
    
    axes[0].plot(per_step_regret, label=name, color=color, alpha=0.5, linewidth=1)
    axes[1].plot(cumulative_regret, label=name, color=color, linewidth=2)

axes[0].set_xlabel('Step')
axes[0].set_ylabel('Per-Step Regret')
axes[0].set_title('Per-Step Regret')
axes[0].legend()

axes[1].set_xlabel('Step')
axes[1].set_ylabel('Cumulative Regret')
axes[1].set_title('Cumulative Regret')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Action Selection Frequency

How often does each strategy pick each arm? We look at the distribution over the final run.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Run a single instance to get action counts
np.random.seed(123)
bandit_single = MultiArmedBandit(k=10)

strat_instances = {
    'Epsilon-Greedy': EpsilonGreedy(10, epsilon=0.1),
    'UCB': UCB(10, c=2.0),
    'Thompson Sampling': ThompsonSampling(10),
}

action_histories = {name: [] for name in strat_instances}

for t in range(5000):
    for name, strat in strat_instances.items():
        a = strat.select_action()
        r = bandit_single.pull(a)
        strat.update(a, r)
        action_histories[name].append(a)

for ax, (name, actions) in zip(axes, action_histories.items()):
    counts = np.bincount(actions, minlength=10)
    bar_colors = ['#e74c3c' if i == bandit_single.optimal_action else '#3498db' for i in range(10)]
    ax.bar(range(10), counts, color=bar_colors)
    ax.set_xlabel('Arm')
    ax.set_ylabel('Times Selected')
    ax.set_title(f'{name}\n(red = optimal arm)')
    ax.set_xticks(range(10))

plt.tight_layout()
plt.show()

## Summary

| Strategy | Key Idea | Pros | Cons |
|----------|----------|------|------|
| **Epsilon-Greedy** | Explore randomly with probability $\epsilon$ | Simple, easy to tune | Explores uniformly (wastes pulls on bad arms) |
| **UCB** | Optimistic in the face of uncertainty | No hyperparameter besides $c$, theoretical guarantees | Deterministic, can be slow to adapt |
| **Thompson Sampling** | Sample from posterior beliefs | Adaptive exploration, strong empirical performance | Requires choosing a prior model |

Thompson Sampling and UCB typically outperform epsilon-greedy because they direct exploration toward uncertain arms rather than exploring uniformly at random.